# CSV to JSON Converter
Bu notebook, atasözleri ve deyimlerin bulunduğu CSV dosyalarını JSON formatına dönüştürür.

## Dosya Yapısı:
- **Input:** `atasozleri_coklu.csv`, `deyimler_coklu.csv`
- **Output:** `public/data/atasozleri.json`, `public/data/deyimler.json`

## CSV Formatı:
```
Deyim/Atasözü;Tanım_No;Tanım;Örnek
```

## JSON Formatı:
```json
[
  {
    "text": "deyim veya atasözü metni",
    "definitions": [
      {
        "id": 1,
        "definition": "tanım metni",
        "example": "örnek cümle"
      }
    ]
  }
]
```

In [1]:
import pandas as pd
import json
import os
from pathlib import Path
from collections import defaultdict

## 1. CSV Okuma ve Dönüştürme Fonksiyonu

In [2]:
def csv_to_json(csv_path: str, json_output_path: str, data_type: str = "deyim"):
    """
    CSV dosyasını JSON formatına dönüştürür.
    
    Args:
        csv_path: CSV dosyasının yolu
        json_output_path: JSON çıktı dosyasının yolu
        data_type: Veri tipi ("deyim" veya "atasözü")
    """
    print(f"\n{'='*60}")
    print(f"📂 {data_type.upper()} Dönüştürülüyor...")
    print(f"{'='*60}")
    
    # CSV dosyasını oku (Türkçe karakterler için UTF-8 encoding)
    try:
        df = pd.read_csv(
            csv_path, 
            sep=';', 
            encoding='utf-8',
            dtype={
                'Deyim': str,
                'Tanım_No': int,
                'Tanım': str,
                'Örnek': str
            },
            keep_default_na=False  # Boş değerleri NaN yapmamak için
        )
        print(f"✅ CSV dosyası başarıyla okundu: {csv_path}")
        print(f"📊 Toplam satır sayısı: {len(df)}")
    except Exception as e:
        print(f"❌ HATA: CSV dosyası okunamadı - {e}")
        return
    
    # İlk sütunun adını normalize et (hem "Deyim" hem "Atasözü" olabilir)
    first_column = df.columns[0]
    df.rename(columns={first_column: 'text'}, inplace=True)
    
    # Boş satırları temizle
    df = df[df['text'].str.strip() != '']
    
    # Aynı deyim/atasözünü grupla
    grouped_data = defaultdict(list)
    
    for _, row in df.iterrows():
        text = row['text'].strip()
        definition_id = int(row['Tanım_No']) if pd.notna(row['Tanım_No']) else 1
        definition = row['Tanım'].strip() if pd.notna(row['Tanım']) else ""
        example = row['Örnek'].strip() if pd.notna(row['Örnek']) else ""
        
        grouped_data[text].append({
            "id": definition_id,
            "definition": definition,
            "example": example
        })
    
    # JSON yapısını oluştur
    json_data = []
    for text, definitions in sorted(grouped_data.items()):
        # Tanımları ID'ye göre sırala
        definitions.sort(key=lambda x: x['id'])
        
        json_data.append({
            "text": text,
            "definitions": definitions
        })
    
    # JSON dosyasını yaz
    try:
        # Çıktı klasörünü oluştur (yoksa)
        output_dir = os.path.dirname(json_output_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            print(f"📁 Klasör oluşturuldu: {output_dir}")
        
        with open(json_output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        
        print(f"✅ JSON dosyası başarıyla oluşturuldu: {json_output_path}")
        print(f"📊 Toplam {data_type} sayısı: {len(json_data)}")
        print(f"📊 Toplam tanım sayısı: {sum(len(item['definitions']) for item in json_data)}")
        
        # İlk 3 örneği göster
        print(f"\n📋 İlk 3 {data_type} örneği:")
        for i, item in enumerate(json_data[:3], 1):
            print(f"\n  {i}. {item['text']}")
            for def_item in item['definitions']:
                print(f"     - Tanım {def_item['id']}: {def_item['definition'][:60]}...")
        
    except Exception as e:
        print(f"❌ HATA: JSON dosyası yazılamadı - {e}")

## 2. Dosya Yollarını Belirle

In [3]:
# Proje kök dizinini bul
current_dir = Path.cwd()
project_root = current_dir.parent.parent if current_dir.name == 'scripts' else current_dir.parent if current_dir.name == 'src' else current_dir

print(f"🏠 Proje kök dizini: {project_root}")

# CSV dosya yolları
atasozleri_csv = project_root / "src" / "scripts" / "atasozleri_coklu.csv"
deyimler_csv = project_root / "src" / "scripts" / "deyimler_coklu.csv"

# JSON çıktı yolları
atasozleri_json = project_root / "public" / "data" / "atasozleri.json"
deyimler_json = project_root / "public" / "data" / "deyimler.json"

print(f"\n📄 CSV Dosyaları:")
print(f"  - Atasözleri: {atasozleri_csv}")
print(f"  - Deyimler: {deyimler_csv}")
print(f"\n📝 JSON Çıktıları:")
print(f"  - Atasözleri: {atasozleri_json}")
print(f"  - Deyimler: {deyimler_json}")

# Dosyaların varlığını kontrol et
print(f"\n✔️ Dosya Kontrolü:")
print(f"  Atasözleri CSV: {'✅ Mevcut' if atasozleri_csv.exists() else '❌ Bulunamadı'}")
print(f"  Deyimler CSV: {'✅ Mevcut' if deyimler_csv.exists() else '❌ Bulunamadı'}")

🏠 Proje kök dizini: d:\Dosya\Kodlar\deyimatasozugame

📄 CSV Dosyaları:
  - Atasözleri: d:\Dosya\Kodlar\deyimatasozugame\src\scripts\atasozleri_coklu.csv
  - Deyimler: d:\Dosya\Kodlar\deyimatasozugame\src\scripts\deyimler_coklu.csv

📝 JSON Çıktıları:
  - Atasözleri: d:\Dosya\Kodlar\deyimatasozugame\public\data\atasozleri.json
  - Deyimler: d:\Dosya\Kodlar\deyimatasozugame\public\data\deyimler.json

✔️ Dosya Kontrolü:
  Atasözleri CSV: ✅ Mevcut
  Deyimler CSV: ✅ Mevcut


## 3. Dönüştürme İşlemini Başlat

In [4]:
# Atasözlerini dönüştür
if atasozleri_csv.exists():
    csv_to_json(str(atasozleri_csv), str(atasozleri_json), "atasözü")
else:
    print(f"⚠️ Atasözleri CSV dosyası bulunamadı: {atasozleri_csv}")


📂 ATASÖZÜ Dönüştürülüyor...
✅ CSV dosyası başarıyla okundu: d:\Dosya\Kodlar\deyimatasozugame\src\scripts\atasozleri_coklu.csv
📊 Toplam satır sayısı: 2783
✅ JSON dosyası başarıyla oluşturuldu: d:\Dosya\Kodlar\deyimatasozugame\public\data\atasozleri.json
📊 Toplam atasözü sayısı: 2256
📊 Toplam tanım sayısı: 2783

📋 İlk 3 atasözü örneği:

  1. Allah bal mumu yakana bal mumu, yağ mumu yakana yağ mumu verir
     - Tanım 1: Tanrı bol harcayana bol, az harcayana az verir....

  2. Allah bana, ben de sana
     - Tanım 1: şimdi sana borcumu ödeyecek param yok, kazanırsam öderim....

  3. Allah bilir ama kul da sezer
     - Tanım 1: bir işin nasıl bir sonuç vereceğini ancak Tanrı bilir ama in...


In [5]:
# Deyimleri dönüştür
if deyimler_csv.exists():
    csv_to_json(str(deyimler_csv), str(deyimler_json), "deyim")
else:
    print(f"⚠️ Deyimler CSV dosyası bulunamadı: {deyimler_csv}")


📂 DEYIM Dönüştürülüyor...
✅ CSV dosyası başarıyla okundu: d:\Dosya\Kodlar\deyimatasozugame\src\scripts\deyimler_coklu.csv
📊 Toplam satır sayısı: 15923
✅ JSON dosyası başarıyla oluşturuldu: d:\Dosya\Kodlar\deyimatasozugame\public\data\deyimler.json
📊 Toplam deyim sayısı: 10518
📊 Toplam tanım sayısı: 15923

📋 İlk 3 deyim örneği:

  1. (aralarında) kan olmak
     - Tanım 1: aralarında kan davası bulunmak....

  2. (at) tırısa kalkmak
     - Tanım 1: tırıs gitmeye başlamak...

  3. (ayakkabı) ayağını vurmak
     - Tanım 1: bir hizmette çok kullanılan kişice, yakınma olarak kullanıla...
     - Tanım 2: ayakkabı ayağını yara etmek....


## 4. Doğrulama - JSON Dosyalarını Kontrol Et

In [6]:
print("\n" + "="*60)
print("🔍 DOĞRULAMA - JSON Dosyaları Kontrol Ediliyor")
print("="*60)

def validate_json(json_path: str, data_type: str):
    """JSON dosyasını doğrular ve istatistiklerini gösterir."""
    if not os.path.exists(json_path):
        print(f"\n❌ {data_type} JSON dosyası bulunamadı: {json_path}")
        return
    
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"\n✅ {data_type.upper()} JSON Dosyası Doğrulandı")
        print(f"   📊 Toplam kayıt: {len(data)}")
        
        # İstatistikler
        total_definitions = sum(len(item['definitions']) for item in data)
        items_with_multiple_defs = sum(1 for item in data if len(item['definitions']) > 1)
        items_with_examples = sum(1 for item in data for d in item['definitions'] if d['example'])
        
        print(f"   📊 Toplam tanım: {total_definitions}")
        print(f"   📊 Birden fazla tanımlı: {items_with_multiple_defs}")
        print(f"   📊 Örnek içeren tanım: {items_with_examples}")
        
        # Dosya boyutu
        file_size = os.path.getsize(json_path) / 1024  # KB
        print(f"   💾 Dosya boyutu: {file_size:.2f} KB")
        
    except Exception as e:
        print(f"\n❌ {data_type} JSON dosyası doğrulanamadı: {e}")

# Her iki dosyayı da doğrula
validate_json(str(atasozleri_json), "atasözleri")
validate_json(str(deyimler_json), "deyimler")

print("\n" + "="*60)
print("✨ Dönüştürme İşlemi Tamamlandı!")
print("="*60)


🔍 DOĞRULAMA - JSON Dosyaları Kontrol Ediliyor

✅ ATASÖZLERI JSON Dosyası Doğrulandı
   📊 Toplam kayıt: 2256
   📊 Toplam tanım: 2783
   📊 Birden fazla tanımlı: 464
   📊 Örnek içeren tanım: 0
   💾 Dosya boyutu: 672.94 KB

✅ DEYIMLER JSON Dosyası Doğrulandı
   📊 Toplam kayıt: 10518
   📊 Toplam tanım: 15923
   📊 Birden fazla tanımlı: 3843
   📊 Örnek içeren tanım: 5565
   💾 Dosya boyutu: 3365.84 KB

✨ Dönüştürme İşlemi Tamamlandı!
